# Notebook 01: Data Collection and Integration

## Project: Predicting Body Mass Index Among U.S. Adults Using Machine Learning.

## Objective

This notebook retrieves and integrates NHANES data across three survey cycles: 2013–2014, 2015–2016, and 2017–2018.
**Integration Pipeline:**
- **Master Table:** The demographic dataset serves as the master participant-level dataset for each cycle.
- **Feature Merging:** Additional questionnaire, examination, and laboratory datasets are merged onto the master table using the unique participant identifier **SEQN**.
- **Harmonisation:** Variables available across all three cycles are aligned and harmonised.
- **Output:** The cycle-specific datasets are combined to create the prepared dataset used for subsequent cleaning, exploratory data analysis (EDA), and BMI prediction.

### Step 1. Confirm that all selected files were downloaded

In [1]:
import os
import pandas as pd

In [2]:
folder = "../data/2017_2018/"

files = [f for f in os.listdir(folder) if f.lower().endswith(".xpt")]

print(len(files))
print(files)

20
['SLQ_J.xpt', 'DPQ_J.xpt', 'GLU_J.xpt', 'BPX_J.xpt', 'BMX_J.xpt', 'HDL_J.xpt', 'TCHOL_J.xpt', 'PAQ_J.xpt', 'MCQ_J.xpt', 'DUQ_J.xpt', 'DEMO_J.xpt', 'INQ_J.xpt', 'INS_J.xpt', 'DIQ_J.xpt', 'GHB_J.xpt', 'ALQ_J.xpt', 'SMQ_J.xpt', 'WHQ_J.xpt', 'BIOPRO_J.xpt', 'TRIGLY_J.xpt']


The folder contains 20 NHANES datasets, confirming that all selected files for the 2017–2018 survey cycle were downloaded successfully.

### Step 2. Load each XPT file into a pandas DataFrame. 
Check the dimension of each dataset to confirm that the files can be read successfully.

In [3]:
import os
import pandas as pd

folder = "../data/2017_2018/"

files = [f for f in os.listdir(folder) if f.lower().endswith(".xpt")]

datasets = {}

for file in files:
    name = file.replace(".xpt", "").replace(".XPT", "")
    datasets[name] = pd.read_sas(os.path.join(folder, file))
    print(name, datasets[name].shape)

SLQ_J (6161, 11)
DPQ_J (5533, 11)
GLU_J (3036, 4)
BPX_J (8704, 21)
BMX_J (8704, 21)
HDL_J (7435, 3)
TCHOL_J (7435, 3)
PAQ_J (5856, 17)
MCQ_J (8897, 76)
DUQ_J (4572, 41)
DEMO_J (9254, 46)
INQ_J (9254, 16)
INS_J (3036, 5)
DIQ_J (8897, 54)
GHB_J (6401, 2)
ALQ_J (5533, 10)
SMQ_J (6724, 37)
WHQ_J (6161, 37)
BIOPRO_J (6401, 41)
TRIGLY_J (3036, 10)


The datasets contain different numbers of participants because some examinations, laboratory tests, and questionnaires were administered only to eligible subsamples. 

### Step 3. Merge the Datasets
The demographic dataset (DEMO_J) was used as the master dataset because it contains all participants in the 2017–2018 survey cycle. The remaining datasets were merged using the unique participant identifier (`SEQN`) with a left join to preserve all participants while adding available demographic, lifestyle, clinical, laboratory, and questionnaire variables.

In [4]:
df_2017_2018 = datasets["DEMO_J"].copy()

for name, data in datasets.items():
    if name != "DEMO_J":
        df_2017_2018 = df_2017_2018.merge(
            data,
            on="SEQN",
            how="left"
        )

print(df_2017_2018.shape)

(9254, 447)


The merged 2017–2018 dataset contains **9,254 participants** and **447 variables**, indicating that all selected datasets were successfully integrated. Missing values are expected because some laboratory tests and questionnaires were administered only to eligible participants.

### Step 4. Verify the merge

In [5]:
# Check dataset dimensions
df_2017_2018.shape

# Check duplicate participants
df_2017_2018["SEQN"].duplicated().sum()

# Check missing values
df_2017_2018.isnull().sum().sort_values(ascending=False).head(20)

BMIHEAD     9254
MCQ230D     9253
SMQ665B     9253
SMQ665D     9252
DIQ175X     9252
SMQ665A     9251
SMQ665C     9250
DIQ175W     9249
MCQ510B     9248
DUQ320      9246
MCQ230C     9245
MCD240C     9245
MCQ510E     9242
SMQ661      9240
DIQ175V     9240
WHD080U     9237
DIQ175R     9235
MCQ510C     9230
BMIRECUM    9230
DUD380F     9229
dtype: int64

These variables have more than **9,200** missing values, meaning they contain data for very few participants (often because they apply only to specific age groups, medical conditions, or questionnaire skip patterns).
For example:
BMIHEAD → Only collected for a small subgroup.
DUQ320 → Asked only of eligible participants.
SMQ665A–D → Smoking questions for a subset.
DIQ175* → Diabetes follow-up questions

### Step 5. Remove Variables with Excessive Missing Values (>= 80% missing values)
This was removed to improve data quality while retaining all study participants.

In [6]:
# Percentage of missing values
missing_percent = df_2017_2018.isnull().mean() * 100

# Keep variables with <= 80% missing
df_2017_2018 = df_2017_2018.loc[:, missing_percent <= 80]

print(df_2017_2018.shape)

(9254, 244)


The merged dataset retained all **9,254 participants**, while the number of variables decreased from **447 to 244** after removing variables with excessive missing data. This reduced the number of variables from 447 to 244 for further processing without reducing the sample size.

### Step 6. Select Variables for Analysis

The merged dataset contains 244 variables after removing variables with excessive missing data. To create a focused dataset for the capstone, variables relevant to BMI and obesity prediction were selected based on epidemiological evidence, clinical relevance, and lifestyle factors.

In [7]:
analysis_vars = [

    # Identifier
    "SEQN",

    # Demographics
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "DMDMARTL",
    "INDFMPIR",

    # Body Measures
    "BMXBMI",
    "BMXWAIST",

    # Blood Pressure
    "BPXSY1",
    "BPXDI1",
    "BPQ020",      # Ever told you had high blood pressure

    # Laboratory
    "LBDHDD",      # HDL
    "LBXTC",       # Total cholesterol
    "LBXGH",       # HbA1c

    # Smoking
    "SMQ020",

    # Alcohol
    "ALQ111",
    "ALQ121",

    # Physical Activity
    "PAQ605",
    "PAQ620",
    "PAD680",
    "PAQ650",      # Walk/Bicycle for transportation

    # Drug Use
    "DUQ200",

    # Medical Conditions
    "DIQ010",
    "DIQ160",      # Prediabetes (if available)
    "MCQ160A",     # Arthritis
    "MCQ160B",     # Congestive heart failure
    "MCQ160C",     # Coronary heart disease
    "MCQ160E",     # Asthma (if available)
    "MCQ160F",     # Stroke (if available)

    # Mental Health
    "DPQ020",

    # Sleep
    "SLD012",
    "SLQ300",      # Sleep disorder/trouble (if available)

    # Weight Behaviour
    "WHQ030"

]

# Keep only variables that actually exist
analysis_vars = [v for v in analysis_vars if v in df_2017_2018.columns]

df_analysis = df_2017_2018[analysis_vars]

df_analysis.shape

(9254, 33)

### Step 7. Verify Missing Variables

In [8]:
df_analysis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9254 entries, 0 to 9253
Data columns (total 33 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      9254 non-null   float64
 1   RIDAGEYR  9254 non-null   float64
 2   RIAGENDR  9254 non-null   float64
 3   RIDRETH3  9254 non-null   float64
 4   DMDEDUC2  5569 non-null   float64
 5   DMDMARTL  5569 non-null   float64
 6   INDFMPIR  8023 non-null   float64
 7   BMXBMI    8005 non-null   float64
 8   BMXWAIST  7601 non-null   float64
 9   BPXSY1    6302 non-null   float64
 10  BPXDI1    6302 non-null   float64
 11  LBDHDD    6738 non-null   float64
 12  LBXTC     6738 non-null   float64
 13  LBXGH     6045 non-null   float64
 14  SMQ020    5856 non-null   float64
 15  ALQ111    5130 non-null   float64
 16  ALQ121    4545 non-null   float64
 17  PAQ605    5856 non-null   float64
 18  PAQ620    5856 non-null   float64
 19  PAD680    5846 non-null   float64
 20  PAQ650    5856 non-null   floa

In [9]:
df_analysis.isnull().sum().sort_values(ascending=False)

DUQ200      6054
ALQ121      4709
DPQ020      4161
ALQ111      4124
MCQ160E     3685
MCQ160A     3685
MCQ160C     3685
MCQ160B     3685
MCQ160F     3685
DMDMARTL    3685
DMDEDUC2    3685
DIQ160      3560
PAD680      3408
PAQ650      3398
SMQ020      3398
PAQ605      3398
PAQ620      3398
LBXGH       3209
SLD012      3141
SLQ300      3093
WHQ030      3093
BPXDI1      2952
BPXSY1      2952
LBXTC       2516
LBDHDD      2516
BMXWAIST    1653
BMXBMI      1249
INDFMPIR    1231
DIQ010       357
RIDAGEYR       0
RIDRETH3       0
RIAGENDR       0
SEQN           0
dtype: int64

### Step 8. Repeat data cleaning with data from 2015 to 2016

In [10]:
import os
import pandas as pd

folder = "/Users/akazong/Data_science online/Capstone/2015_2016/"

files = [f for f in os.listdir(folder) if f.lower().endswith(".xpt")]

print(len(files))
print(files)

20
['SLQ_I.xpt', 'DPQ_I.xpt', 'GLU_I.xpt', 'BPX_I.xpt', 'MCQ_I.xpt', 'BMX_I.xpt', 'HDL_I.xpt', 'TCHOL_I.xpt', 'PAQ_I.xpt', 'DUQ_I.xpt', 'DEMO_I.xpt', 'INS_I.xpt', 'DIQ_I.xpt', 'GHB_I.xpt', 'ALQ_I.xpt', 'SMQ_I.xpt', 'INQ_I.xpt', 'WHQ_I.xpt', 'BIOPRO_I.xpt', 'TRIGLY_I.xpt']


In [11]:
# Load each XPT file into a pandas DataFrame
datasets_1 = {}

for file in files:
    name = file.replace(".xpt", "").replace(".XPT", "")
    datasets_1[name] = pd.read_sas(os.path.join(folder, file))
    print(name, datasets_1[name].shape)

SLQ_I (6327, 8)
DPQ_I (5735, 11)
GLU_I (3191, 4)
BPX_I (9544, 21)
MCQ_I (9575, 90)
BMX_I (9544, 26)
HDL_I (8021, 3)
TCHOL_I (8021, 3)
PAQ_I (9255, 94)
DUQ_I (4843, 42)
DEMO_I (9971, 47)
INS_I (3191, 7)
DIQ_I (9575, 54)
GHB_I (6744, 2)
ALQ_I (5735, 10)
SMQ_I (7001, 42)
INQ_I (9971, 16)
WHQ_I (6327, 37)
BIOPRO_I (6744, 38)
TRIGLY_I (3191, 6)


In [12]:
# Merge the Datasets
df_2015_2016 = datasets_1["DEMO_I"].copy()

for name, data in datasets_1.items():
    if name != "DEMO_I":
        df_2015_2016 = df_2015_2016.merge(
            data,
            on="SEQN",
            how="left"
        )

# Check dataset dimensions
df_2015_2016.shape

# Check duplicate participants
df_2015_2016["SEQN"].duplicated().sum()

# Check missing values
df_2015_2016.isnull().sum().sort_values(ascending=False).head(20)

MCQ230D     9971
MCQ240D     9971
MCQ240I     9971
MCQ240K     9971
MCQ240R     9971
MCQ240Y     9971
BMIHEAD     9971
DIQ175X     9970
MCQ240DK    9970
MCQ240B     9969
MCQ240C     9969
MCQ240H     9969
MCQ240V     9969
PAQ724M     9968
MCQ240T     9968
SMQ665B     9968
SMQ665D     9968
MCQ240Q     9967
MCQ240AA    9967
MCQ240Z     9967
dtype: int64

In [13]:
# Select the same variables as in step 6 for analysis
analysis_vars = [

    # Identifier
    "SEQN",

    # Demographics
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "DMDMARTL",
    "INDFMPIR",

    # Body Measures
    "BMXBMI",
    "BMXWAIST",

    # Blood Pressure
    "BPXSY1",
    "BPXDI1",
    "BPQ020",      # Ever told you had high blood pressure

    # Laboratory
    "LBDHDD",      # HDL
    "LBXTC",       # Total cholesterol
    "LBXGH",       # HbA1c

    # Smoking
    "SMQ020",

    # Alcohol
    "ALQ111",
    "ALQ121",

    # Physical Activity
    "PAQ605",
    "PAQ620",
    "PAD680",
    "PAQ650",      # Walk/Bicycle for transportation

    # Drug Use
    "DUQ200",

    # Medical Conditions
    "DIQ010",
    "DIQ160",      # Prediabetes (if available)
    "MCQ160A",     # Arthritis
    "MCQ160B",     # Congestive heart failure
    "MCQ160C",     # Coronary heart disease
    "MCQ160E",     # Asthma (if available)
    "MCQ160F",     # Stroke (if available)

    # Mental Health
    "DPQ020",

    # Sleep
    "SLD012",
    "SLQ300",      # Sleep disorder/trouble (if available)

    # Weight Behaviour
    "WHQ030"

]

# Keep only variables that actually exist
analysis_vars = [v for v in analysis_vars if v in df_2015_2016.columns]

df_analysis = df_2015_2016[analysis_vars]

df_analysis.shape

(9971, 31)

In [14]:
# Verify missing variables
df_analysis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9971 entries, 0 to 9970
Data columns (total 31 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      9971 non-null   float64
 1   RIDAGEYR  9971 non-null   float64
 2   RIAGENDR  9971 non-null   float64
 3   RIDRETH3  9971 non-null   float64
 4   DMDEDUC2  5719 non-null   float64
 5   DMDMARTL  5719 non-null   float64
 6   INDFMPIR  8919 non-null   float64
 7   BMXBMI    8756 non-null   float64
 8   BMXWAIST  8313 non-null   float64
 9   BPXSY1    7145 non-null   float64
 10  BPXDI1    7145 non-null   float64
 11  LBDHDD    7256 non-null   float64
 12  LBXTC     7256 non-null   float64
 13  LBXGH     6326 non-null   float64
 14  SMQ020    5992 non-null   float64
 15  PAQ605    6966 non-null   float64
 16  PAQ620    6965 non-null   float64
 17  PAD680    6951 non-null   float64
 18  PAQ650    6963 non-null   float64
 19  DUQ200    3428 non-null   float64
 20  DIQ010    9575 non-null   floa

In [15]:
df_analysis.isnull().sum().sort_values(ascending=False)

DUQ200      6543
DPQ020      4807
MCQ160C     4252
DMDEDUC2    4252
DMDMARTL    4252
MCQ160F     4252
MCQ160E     4252
MCQ160B     4252
MCQ160A     4252
SMQ020      3979
DIQ160      3926
SLD012      3677
LBXGH       3645
WHQ030      3644
SLQ300      3644
PAD680      3020
PAQ650      3008
PAQ620      3006
PAQ605      3005
BPXDI1      2826
BPXSY1      2826
LBXTC       2715
LBDHDD      2715
BMXWAIST    1658
BMXBMI      1215
INDFMPIR    1052
DIQ010       396
RIDAGEYR       0
RIDRETH3       0
RIAGENDR       0
SEQN           0
dtype: int64

### Step 9. Repeat data cleaning with data from 2013 to 2014

In [16]:
folder = "/Users/akazong/Data_science online/Capstone/2013_2014/"

files = [f for f in os.listdir(folder) if f.lower().endswith(".xpt")]

print(len(files))
print(files)

20
['GLU_H.xpt', 'DPQ_H.xpt', 'SLQ_H.xpt', 'BPX_H.xpt', 'MCQ_H.xpt', 'PAQ_H.xpt', 'TCHOL_H.xpt', 'HDL_H.xpt', 'BMX_H.xpt', 'DEMO_H.xpt', 'DUQ_H.xpt', 'SMQ_H.xpt', 'ALQ_H.xpt', 'GHB_H.xpt', 'DIQ_H.xpt', 'INS_H.xpt', 'INQ_H.xpt', 'TRIGLY_H.xpt', 'BIOPRO_H.xpt', 'WHQ_H.xpt']


In [17]:
# Load each XPT file into a pandas DataFrame
datasets_2 = {}

for file in files:
    name = file.replace(".xpt", "").replace(".XPT", "")
    datasets_2[name] = pd.read_sas(os.path.join(folder, file))
    print(name, datasets_2[name].shape)

GLU_H (3329, 6)
DPQ_H (5924, 11)
SLQ_H (6464, 4)
BPX_H (9813, 23)
MCQ_H (9770, 95)
PAQ_H (9484, 96)
TCHOL_H (8291, 3)
HDL_H (8291, 3)
BMX_H (9813, 26)
DEMO_H (10175, 47)
DUQ_H (5057, 42)
SMQ_H (7168, 32)
ALQ_H (5924, 10)
GHB_H (6979, 2)
DIQ_H (9770, 54)
INS_H (3329, 6)
INQ_H (10175, 15)
TRIGLY_H (3329, 6)
BIOPRO_H (6979, 38)
WHQ_H (6464, 34)


In [18]:
# Merge the Datasets
df_2013_2014 = datasets_2["DEMO_H"].copy()

for name, data in datasets_2.items():
    if name != "DEMO_H":
        df_2013_2014 = df_2013_2014.merge(
            data,
            on="SEQN",
            how="left"
        )

# Check dataset dimensions
df_2013_2014.shape

# Check duplicate participants
df_2013_2014["SEQN"].duplicated().sum()

# Check missing values
df_2013_2014.isnull().sum().sort_values(ascending=False).head(20)

BMIHEAD     10175
MCQ240I     10175
MCQ240R     10175
PAQ759T     10174
MCQ240V     10174
SMQ665D     10173
MCQ230D     10173
MCQ240H     10173
MCQ240C     10173
MCQ240T     10172
DIQ175W     10172
PAQ759V     10172
MCQ240D     10172
PAQ724L     10172
MCQ240AA    10172
MCQ240Y     10172
MCQ240Z     10172
SMQ665B     10172
MCQ240K     10171
MCQ240B     10171
dtype: int64

In [19]:
# Select the same variables as in step 6 for analysis
analysis_vars = [

    # Identifier
    "SEQN",

    # Demographics
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "DMDMARTL",
    "INDFMPIR",

    # Body Measures
    "BMXBMI",
    "BMXWAIST",

    # Blood Pressure
    "BPXSY1",
    "BPXDI1",
    "BPQ020",      # Ever told you had high blood pressure

    # Laboratory
    "LBDHDD",      # HDL
    "LBXTC",       # Total cholesterol
    "LBXGH",       # HbA1c

    # Smoking
    "SMQ020",

    # Alcohol
    "ALQ111",
    "ALQ121",

    # Physical Activity
    "PAQ605",
    "PAQ620",
    "PAD680",
    "PAQ650",      # Walk/Bicycle for transportation

    # Drug Use
    "DUQ200",

    # Medical Conditions
    "DIQ010",
    "DIQ160",      # Prediabetes (if available)
    "MCQ160A",     # Arthritis
    "MCQ160B",     # Congestive heart failure
    "MCQ160C",     # Coronary heart disease
    "MCQ160E",     # Asthma (if available)
    "MCQ160F",     # Stroke (if available)

    # Mental Health
    "DPQ020",

    # Sleep
    "SLD012",
    "SLQ300",      # Sleep disorder/trouble (if available)

    # Weight Behaviour
    "WHQ030"

]

# Keep only variables that actually exist
analysis_vars = [v for v in analysis_vars if v in df_2013_2014.columns]

df_analysis = df_2013_2014[analysis_vars]

df_analysis.shape

(10175, 29)

In [20]:
# Verify missing variables
df_analysis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10175 entries, 0 to 10174
Data columns (total 29 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      10175 non-null  float64
 1   RIDAGEYR  10175 non-null  float64
 2   RIAGENDR  10175 non-null  float64
 3   RIDRETH3  10175 non-null  float64
 4   DMDEDUC2  5769 non-null   float64
 5   DMDMARTL  5769 non-null   float64
 6   INDFMPIR  9390 non-null   float64
 7   BMXBMI    9055 non-null   float64
 8   BMXWAIST  8661 non-null   float64
 9   BPXSY1    7172 non-null   float64
 10  BPXDI1    7172 non-null   float64
 11  LBDHDD    7624 non-null   float64
 12  LBXTC     7624 non-null   float64
 13  LBXGH     6643 non-null   float64
 14  SMQ020    6113 non-null   float64
 15  PAQ605    7148 non-null   float64
 16  PAQ620    7148 non-null   float64
 17  PAD680    7139 non-null   float64
 18  PAQ650    7147 non-null   float64
 19  DUQ200    3701 non-null   float64
 20  DIQ010    9769 non-null   fl

In [21]:
df_analysis.isnull().sum().sort_values(ascending=False)

DUQ200      6474
DPQ020      4779
MCQ160F     4406
DMDEDUC2    4406
DMDMARTL    4406
MCQ160E     4406
MCQ160C     4406
MCQ160B     4406
MCQ160A     4406
SMQ020      4062
DIQ160      3888
WHQ030      3711
LBXGH       3532
PAD680      3036
PAQ650      3028
PAQ605      3027
PAQ620      3027
BPXDI1      3003
BPXSY1      3003
LBXTC       2551
LBDHDD      2551
BMXWAIST    1514
BMXBMI      1120
INDFMPIR     785
DIQ010       406
RIDAGEYR       0
RIDRETH3       0
RIAGENDR       0
SEQN           0
dtype: int64

### 10. Combine NHANES Survey Cycles
After selecting the analysis variables, the three survey cycles (2013–2014, 2015–2016, and 2017–2018) were combined into a single master dataset. Only variables present in all three cycles were retained to ensure a consistent data structure. A new variable (`Survey_Cycle`) was added to identify the original survey cycle for each participant. The final merged dataset was then checked for duplicate participant identifiers and the distribution of participants across survey cycles.

In [22]:
# Step 1: Identify variables common to all three survey cycles
common_cols = list(
    set(df_2017_2018.columns)
    & set(df_2015_2016.columns)
    & set(df_2013_2014.columns)
)

print("Number of common variables:", len(common_cols))
# print(sorted(common_cols))  # Run if lecturer requires it

# Step 2: Keep only the common variables
# This ensures all datasets have identical columns
# Combine into one master dataset

df_2017_2018 = df_2017_2018[common_cols].copy()
df_2015_2016 = df_2015_2016[common_cols].copy()
df_2013_2014 = df_2013_2014[common_cols].copy()

# Step 3: Add a survey cycle identifier
# This allows us to identify the original survey cycle after all datasets have been merged

df_2017_2018["Survey_Cycle"] = "2017-2018"
df_2015_2016["Survey_Cycle"] = "2015-2016"
df_2013_2014["Survey_Cycle"] = "2013-2014"

# Step 4: Combine the three survey cycles
# The datasets are stacked vertically because they contain the same variables
# These variables are measured in different survey years

df_merge = pd.concat(
    [df_2013_2014,
     df_2015_2016,
     df_2017_2018],
    ignore_index=True
)

# Step 5: Verify the combined dataset
# Check the number of rows and columns after merging

print("Dataset shape:", df_merge.shape)

# Step 6: Check for duplicate participants
# Each NHANES participant should have a unique SEQN within the combined dataset

print("Duplicate SEQN values:",
      df_merge["SEQN"].duplicated().sum())

# Step 7: Verify participants from each survey cycle
# This confirms that all three survey cycles were merged successfully

print(df_merge["Survey_Cycle"].value_counts())

Number of common variables: 197
Dataset shape: (29400, 198)
Duplicate SEQN values: 0
Survey_Cycle
2013-2014    10175
2015-2016     9971
2017-2018     9254
Name: count, dtype: int64


In [23]:
# 10. Save the combined dataset as .csv for ease of use in the next section

In [24]:
df_merge.to_csv("NHANES_Combined.csv", index=False)

In [25]:
# Select the same variables as in step 6 for analysis
analysis_vars = [

    # Identifier
    "SEQN",

    # Demographics
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "DMDMARTL",
    "INDFMPIR",

    # Body Measures
    "BMXBMI",
    "BMXWAIST",

    # Blood Pressure
    "BPXSY1",
    "BPXDI1",
    "BPQ020",      # Ever told you had high blood pressure

    # Laboratory
    "LBDHDD",      # HDL
    "LBXTC",       # Total cholesterol
    "LBXGH",       # HbA1c

    # Smoking
    "SMQ020",

    # Alcohol
    "ALQ111",
    "ALQ121",

    # Physical Activity
    "PAQ605",
    "PAQ620",
    "PAD680",
    "PAQ650",      # Walk/Bicycle for transportation

    # Drug Use
    "DUQ200",

    # Medical Conditions
    "DIQ010",
    "DIQ160",      # Prediabetes (if available)
    "MCQ160A",     # Arthritis
    "MCQ160B",     # Congestive heart failure
    "MCQ160C",     # Coronary heart disease
    "MCQ160E",     # Asthma (if available)
    "MCQ160F",     # Stroke (if available)

    # Mental Health
    "DPQ020",

    # Sleep
    "SLD012",
    "SLQ300",      # Sleep disorder/trouble (if available)

    # Weight Behaviour
    "WHQ030"

]

# Keep only variables that actually exist
analysis_vars = [v for v in analysis_vars if v in df_merge.columns]

df_final = df_merge[analysis_vars]

df_final.shape

(29400, 29)

In [26]:
df_final.to_csv("../outputs/Final_edited.csv", index=False)

### Step 11: Check the final output

In [27]:
print("Final dataset shape:", df_final.shape)
print("Number of unique participants:", df_final["SEQN"].nunique())
print("Duplicate participant IDs:", df_final["SEQN"].duplicated().sum())

Final dataset shape: (29400, 29)
Number of unique participants: 29400
Duplicate participant IDs: 0
